In [16]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

In [17]:
task = '''
        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       '''


In [22]:
from autogen_agentchat.agents import AssistantAgent

writer = AssistantAgent(
    name="Writer",
    model_client=llm_config,
    system_message=(
        "You are a writer. You write engaging and concise "
        "blog posts (with title) on given topics. You must polish "
        "your writing based on the feedback you receive and give a refined "
        "version. Only return your final work without additional comments."
    ),
)

In [23]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gemini-3.8-flash",
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
        "structured_output": True,
    },
)

In [25]:
critic = AssistantAgent(
    name="Critic",
    model_client=model_client,
    system_message=(
        "You are a critic. You review the work of the writer "
        "and provide constructive feedback to help improve "
        "the quality of the content."
    ),
)

In [28]:
res = await critic.run(
    task=task
)

print(res.messages[-1].content)

**Title: The Dawn of Hope AI**

Can artificial intelligence do more than optimize spreadsheets? Enter **Hope AI**—a shift from machine efficiency to human elevation. 

While headlines often amplify fears of automation, Hope AI directs cutting-edge models toward our most urgent crises: accelerating cancer research, modeling climate resilience, and democratizing world-class education. It isn’t merely about building faster algorithms; it’s about infusing technological power with ethical purpose. 

Technology is a mirror of human intent. With Hope AI, the horizon isn't a dystopian question mark—it’s an inspiring, human-first reality. The future isn't just intelligent; it's hopeful.


In [29]:
SEO_reviewer = AssistantAgent(
    name="SEO_Reviewer",
    model_client=model_client,
    system_message=(
        "You are an SEO reviewer, known for your ability to optimize "
        "content for search engines, ensuring that it ranks well "
        "and attracts organic traffic. "
        "Make sure your suggestions are concise (within 3 bullet points), "
        "concrete, and to the point. "
        "Begin the review by stating your role."
    ),
)


In [31]:
legal_reviewer = AssistantAgent(
    name="Legal_Reviewer",
    model_client=model_client,
    system_message="You are a legal reviewer, known for "
        "your ability to ensure that content is legally compliant "
        "and free from any potential legal issues. "
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role.",
)

In [32]:
ethics_reviewer = AssistantAgent(
    name="Ethics_Reviewer",
    model_client=model_client,
    system_message="You are an ethics reviewer, known for "
        "your ability to ensure that content is ethically sound "
        "and free from any potential ethical issues. " 
        "Make sure your suggestion is concise (within 3 bullet points), "
        "concrete and to the point. "
        "Begin the review by stating your role. ",
)

In [33]:
meta_reviewer = AssistantAgent(
    name="Meta_Reviewer",
    model_client=model_client,
    system_message="You are a meta reviewer, you aggragate and review "
    "the work of other reviewers and give a final suggestion on the content.",
)

In [34]:
def reflection_message(recipient, messages, sender, config):
    return f'''Review the following content. 
            \n\n {recipient.chat_messages_for_summary(sender)[-1]['content']}'''

review_chats = [
    {
     "recipient": SEO_reviewer, 
     "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}. Here Reviewer should be your role",},
     "max_turns": 1},
    {
    "recipient": legal_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'Reviewer': '', 'Review': ''}.",},
     "max_turns": 1},
    {"recipient": ethics_reviewer, "message": reflection_message, 
     "summary_method": "reflection_with_llm",
     "summary_args": {"summary_prompt" : 
        "Return review into as JSON object only:"
        "{'reviewer': '', 'review': ''}",},
     "max_turns": 1},
     {"recipient": meta_reviewer, 
      "message": "Aggregrate feedback from all reviewers and give final suggestions on the writing.", 
     "max_turns": 1},
]


In [39]:
print(type(model_client))

<class 'autogen_ext.models.openai._openai_client.OpenAIChatCompletionClient'>


**Note**: You might get a slightly different response than what's shown in the video. Feel free to try different task.

In [43]:
from autogen_agentchat.agents import AssistantAgent

writer = AssistantAgent(
    name="Writer",
    model_client=model_client,
    system_message="You are a professional content writer."
)

critic = AssistantAgent(
    name="Critic",
    model_client=model_client,
    system_message="You review the writer's work and provide constructive feedback."
)

SEO_reviewer = AssistantAgent(
    name="SEO_Reviewer",
    model_client=model_client,
    system_message="You are an SEO reviewer. Provide three concise and actionable suggestions."
)

In [42]:
print("Model client:", type(model_client))
print("Writer:", type(writer._model_client))
print("Critic:", type(critic._model_client))
print("SEO Reviewer:", type(SEO_reviewer._model_client))

Model client: <class 'autogen_ext.models.openai._openai_client.OpenAIChatCompletionClient'>
Writer: <class 'dict'>
Critic: <class 'autogen_ext.models.openai._openai_client.OpenAIChatCompletionClient'>
SEO Reviewer: <class 'autogen_ext.models.openai._openai_client.OpenAIChatCompletionClient'>


In [44]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

team = RoundRobinGroupChat(
    participants=[writer, critic, SEO_reviewer],
    termination_condition=MaxMessageTermination(3)
)

result = await team.run(task=task)

for message in result.messages:
    print(f"\n{message.source}:")
    print(message.content)


user:

        Write a concise but engaging blogpost about
      Hope AI . Make sure the blogpost is
       within 100 words.
       

Writer:
**Meet Hope AI: Technology with a Heart**

Can artificial intelligence do more than crunch numbers? Enter Hope AI—a vision transforming algorithms into instruments of human progress. 

Designed to tackle real-world challenges, Hope AI bridges the gap between powerful data and genuine empathy. From personalizing education to advancing accessible healthcare, it isn’t built to replace human potential; it’s designed to unlock it. 

In an era of digital noise, Hope AI is a refreshing reminder that innovation should heal, inspire, and uplift. The future of technology isn’t just intelligent—it’s full of hope. 

Are you ready to build a better tomorrow?

Critic:
Here is a review of your blog post:

### **Strengths**
* **Strict Adherence to Constraints:** At **93 words** (including the title), the draft fits comfortably beneath the 100-word limit withou

In [47]:
summary = []

summary.append(res.messages[-1].content)

print(summary[0])

**Title: The Dawn of Hope AI**

Can artificial intelligence do more than optimize spreadsheets? Enter **Hope AI**—a shift from machine efficiency to human elevation. 

While headlines often amplify fears of automation, Hope AI directs cutting-edge models toward our most urgent crises: accelerating cancer research, modeling climate resilience, and democratizing world-class education. It isn’t merely about building faster algorithms; it’s about infusing technological power with ethical purpose. 

Technology is a mirror of human intent. With Hope AI, the horizon isn't a dystopian question mark—it’s an inspiring, human-first reality. The future isn't just intelligent; it's hopeful.
